In [14]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
STREAM_HOST = os.environ.get("SOCKET_HOST", "localhost")
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-structured-streaming-preparation").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64
Spark version: 4.1.2


# Stream Aggregations, Joins, and Time Windows

Beyond filtering, Structured Streaming supports **stateful operations**:  
running counts, word totals, joins -- all updated incrementally as new data arrives.

**Important constraint for aggregations**:
- `append` mode is *not* allowed for unbounded aggregations (without watermarks)
- Use `complete` (full result each micro-batch) or `update` (only changed rows)

## Socket Setup (for the interactive examples)
```bash
# macOS/Linux
nc -lk 11111

# Windows
ncat -lk 11111
```

> **Notebook note**: Socket queries block with `awaitTermination()`.  
> The code cells show the patterns -- uncomment `start()` / `awaitTermination()` to run as a script.

References:
- https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#window-operations-on-event-time

# Example 1: Streaming Count

Count the total number of lines received so far.  
This is a simple stateful aggregation -- a running total that grows each micro-batch.

Because it aggregates over the entire stream, we must use `complete` output mode.

In [4]:
input_stream = (spark.readStream
    .format("socket")
    .option("host", STREAM_HOST)
    .option("port", 11111)
    .load())

count_df = input_stream.select(F.count(F.col("value")).alias("total_lines"))

output = count_df.writeStream.format("console").outputMode("complete")

# NOTE: uncomment to run
query = output.start()
query.awaitTermination(timeout=10)
query.stop()
print("Count query defined -- outputMode='complete' required for aggregations")

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+
|total_lines|
+-----------+
|          0|
+-----------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+
|total_lines|
+-----------+
|          1|
+-----------+

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+
|total_lines|
+-----------+
|          2|
+-----------+

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+
|total_lines|
+-----------+
|          3|
+-----------+

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+
|total_lines|
+-----------+
|          4|
+-----------+

Count query defined -- outputMode='complete' required for aggregations


# Example 2: Streaming Word Count

Split each incoming line into words and count occurrences.  
This is the streaming version of the batch WordCount example from `spark-introduction`.

`F.explode(F.split(...))` creates one row per word from each input line.

In [3]:
input_stream = (spark.readStream
    .format("socket")
    .option("host", STREAM_HOST)
    .option("port", 11111)
    .load())

word_count_df = (input_stream
    .withColumn("word", F.explode(F.split(F.lower(F.col("value")), r"\W+")))
    .where(F.length(F.col("word")) > 0)
    .groupBy("word")
    .count()
    .orderBy(F.col("count").desc()))

output = word_count_df.writeStream.format("console").outputMode("complete")

query = output.start()
query.awaitTermination(timeout=30)
query.stop()
print("WordCount stream query defined -- send text via netcat to see counts")

-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
+----+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|there|    1|
|   hi|    1|
+-----+-----+



-------------------------------------------
Batch: 2
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|there|    2|
|   hi|    2|
+-----+-----+



-------------------------------------------
Batch: 3
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|there|    2|
|   hi|    2|
|    i|    1|
| here|    1|
|   am|    1|
+-----+-----+



-------------------------------------------
Batch: 4
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|there|    3|
|   hi|    3|
|    i|    1|
| here|    1|
|   am|    1|
+-----+-----+



-------------------------------------------
Batch: 5
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|there|    3|
|   hi|    3|
|  bye|    1|
|    i|    1|
| here|    1|
| good|    1|
|   am|    1|
+-----+-----+



[Stage 31:======================================>              (147 + 10) / 200]

WordCount stream query defined -- send text via netcat to see counts


# Example 3: Stream aggregation into files



In [18]:
# IOT sensor stream pattern
# Data format sent via socket: sensorID,timestamp,temperature
# Example lines
#1,2026-06-26 22:00:00,23.5
#1,2026-06-26 22:00:02,24.1

input_stream = (spark.readStream
    .format("socket").option("host", STREAM_HOST).option("port", 11111).load())

# Parse CSV-formatted sensor readings
step1 = (input_stream
    .selectExpr("from_csv(value, '`sensorID` INT, `timestamp` TIMESTAMP, `temperature` DOUBLE') AS r")
    .select("r.*"))

step1.printSchema()

# Windowed aggregation with watermark
step2 = (step1
    ###.withWatermark("timestamp", "10 seconds")     ## this is needed     # late data tolerance
    .groupBy(
        F.col("sensorID"),
        F.window(F.col("timestamp"), "5 seconds"))     # 5-second tumbling windows
    .avg("temperature")
    .select(
        F.col("sensorID"),
        F.expr("window.*"),                            # expands to window.start, window.end
        F.col("avg(temperature)")))

# append mode is valid because watermark bounds the state
output = (step2.writeStream
    .format("csv")
    .outputMode("append")
    .option("path", "out/avg_csv")
    .option("checkpointLocation", "tmp/chk") )

query = output.start()
query.awaitTermination(timeout=30)

import time
time.sleep(30)
query.stop()

print("Straming Aggregation to file done")

root
 |-- sensorID: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature: double (nullable = true)



26/06/27 02:17:46 ERROR FileFormatWriter: Aborting job 88a2008e-a437-40c2-9e45-a619e5764a7a.
java.net.ConnectException: Connection refused
	at java.base/sun.nio.ch.Net.connect0(Native Method)
	at java.base/sun.nio.ch.Net.connect(Net.java:591)
	at java.base/sun.nio.ch.Net.connect(Net.java:580)
	at java.base/sun.nio.ch.NioSocketImpl.connect(NioSocketImpl.java:593)
	at java.base/java.net.SocksSocketImpl.connect(SocksSocketImpl.java:327)
	at java.base/java.net.Socket.connect(Socket.java:633)
	at java.base/java.net.Socket.connect(Socket.java:583)
	at java.base/java.net.Socket.<init>(Socket.java:507)
	at java.base/java.net.Socket.<init>(Socket.java:287)
	at org.apache.spark.sql.execution.streaming.sources.TextSocketMicroBatchStream.initialize(TextSocketMicroBatchStream.scala:72)
	at org.apache.spark.sql.execution.streaming.sources.TextSocketMicroBatchStream.planInputPartitions(TextSocketMicroBatchStream.scala:118)
	at org.apache.spark.sql.execution.datasources.v2.MicroBatchScanExec.inputPart

StreamingQueryException: [STREAM_FAILED] Query [id = 094326ec-7fc4-4446-a094-9d0752034f58, runId = 294cd231-8a00-4283-af30-745bb7659eef] terminated with exception: Connection refused SQLSTATE: XXKST
=== Streaming Query ===
Identifier: [id = 094326ec-7fc4-4446-a094-9d0752034f58, runId = 294cd231-8a00-4283-af30-745bb7659eef]
Current Committed Offsets: {TextSocketV2[host: host.docker.internal, port: 11111]: 3}
Current Available Offsets: {TextSocketV2[host: host.docker.internal, port: 11111]: 4}

Current State: ACTIVE
Thread State: RUNNABLE

Logical Plan:
~WriteToMicroBatchDataSourceV1 FileSink[file:/app/work/data-engineering-preparation/out/avg_csv], 094326ec-7fc4-4446-a094-9d0752034f58, [path=out/avg_csv, checkpointLocation=tmp/chk], Append
+- ~Project [sensorID#366, window#374-T10000ms.start AS start#375, window#374-T10000ms.end AS end#376, avg(temperature)#373]
   +- ~Aggregate [sensorID#366, window#374-T10000ms], [sensorID#366, window#374-T10000ms, avg(temperature#368) AS avg(temperat

# Example 3: Stream–Static DataFrame Join

A **streaming** DataFrame can be joined with a **static** (batch) DataFrame.  
The static DataFrame is broadcast to all workers once; it is looked up for each micro-batch.

Useful for enriching streams with reference data (zip codes, product names, user profiles).

In [5]:
# Static data -- loaded from disk once
courses_df     = spark.read.option("inferSchema","true").json("data/courses.json")
instructors_df = spark.read.option("inferSchema","true").json("data/instructors.json")

# Batch join for reference
join_cond = courses_df["currentInstructor"] == instructors_df["id"]
courses_df.join(instructors_df, join_cond, "inner").show()

+---+-----------------+-----+------+--------------------+-------+---+-----------------+
|cid|currentInstructor|major|number|               title|courses| id|             name|
+---+-----------------+-----+------+--------------------+-------+---+-----------------+
|  0|                4| MIST|  4600|Computer Programm...|    [0]|  4|      Karen Aguar|
|  1|                3| MIST|  4610|Data Managment an...| [0, 1]|  3|Nikhil Srinivasan|
|  2|                1| MIST|  4630|Network-Based App...|    [2]|  1|     Craig Piercy|
|  3|                0| MIST|  5730|Advanced Data Man...| [0, 3]|  0|      Hani Safadi|
+---+-----------------+-----+------+--------------------+-------+---+-----------------+



In [8]:
# Streaming join pattern
# Courses arrive as JSON strings via socket
input_stream = (spark.readStream 
    .format("socket").option("host", STREAM_HOST).option("port", 11111).load())

# Parse JSON
streamed_courses = (input_stream
    .select(F.from_json(F.col("value"), courses_df.schema).alias("c"))
    .select("c.*"))

# Join stream with static instructors
enriched = streamed_courses.join(
    instructors_df,
    streamed_courses["currentInstructor"] == instructors_df["id"],
    "inner")

output = enriched.writeStream.format("console").outputMode("append")

query = output.start()
query.awaitTermination(timeout=30)
query.stop()
print("Stream-DF join complete")

-------------------------------------------
Batch: 0
-------------------------------------------
+---+-----------------+-----+------+-----+-------+---+----+
|cid|currentInstructor|major|number|title|courses| id|name|
+---+-----------------+-----+------+-----+-------+---+----+
+---+-----------------+-----+------+-----+-------+---+----+

-------------------------------------------
Batch: 1
-------------------------------------------
+---+-----------------+-----+------+--------------------+-------+---+-----------------+
|cid|currentInstructor|major|number|               title|courses| id|             name|
+---+-----------------+-----+------+--------------------+-------+---+-----------------+
|  0|                4| MIST|  4600|Computer Programm...|    [0]|  4|      Karen Aguar|
|  1|                3| MIST|  4610|Data Managment an...| [0, 1]|  3|Nikhil Srinivasan|
+---+-----------------+-----+------+--------------------+-------+---+-----------------+

------------------------------------

# Example 4: Stream–Stream Join

Two **streaming** DataFrames can be joined with each other.  
Both sides arrive via separate socket connections and are parsed from JSON.

Key differences from a stream–static join:
- Both sides accumulate state in memory until a matching row arrives on the other side.
- Without watermarks, state is kept **indefinitely** (unbounded memory growth).
- Adding `.withWatermark()` to both sides bounds the state and enables `append` output mode.

```bash
# Requires two netcat servers running simultaneously:
nc -lk 11111   # courses stream
nc -lk 22222   # instructors stream
```

Paste one JSON object per line into each terminal, e.g.:  
Port 11111 → `{"cid":0,"number":"4600","major":"MIST","title":"Computer Programming in Business", "currentInstructor":4}`  
Port 22222 → `{"id":0,"name":"Hani Safadi","courses":[0, 3]}`  
Spark will inner-join on `currentInstructor == id` and print the enriched row.


In [9]:
# ── Example 4b: Stream–Stream Join ───────────────────────────────────────────
# Two socket streams are joined on a key column.
# Port 11111 sends course JSON; port 22222 sends instructor JSON.
# Both streams must have watermarks when joining on event time,
# but for an inner join on a non-time key Spark allows it without watermarks
# (though results accumulate in unbounded state).

# Schema from the static files (reuse what we already loaded)
# courses_df and instructors_df must be loaded first (see Example 3)

streamed_courses = (spark.readStream
    .format("socket").option("host", STREAM_HOST).option("port", 11111).load()
    .select(F.from_json(F.col("value"), courses_df.schema).alias("c"))
    .select("c.*"))

streamed_instructors = (spark.readStream
    .format("socket").option("host", STREAM_HOST).option("port", 22222).load()
    .select(F.from_json(F.col("value"), instructors_df.schema).alias("i"))
    .select("i.*"))

join_condition = streamed_courses["currentInstructor"] == streamed_instructors["id"]
joined_stream = streamed_courses.join(streamed_instructors, join_condition, "inner")

output = joined_stream.writeStream.format("console").outputMode("append")
query = output.start()
query.awaitTermination(timeout=30)
query.stop()
print("Stream-stream join complete")

-------------------------------------------
Batch: 0
-------------------------------------------
+---+-----------------+-----+------+-----+-------+---+----+
|cid|currentInstructor|major|number|title|courses| id|name|
+---+-----------------+-----+------+-----+-------+---+----+
+---+-----------------+-----+------+-----+-------+---+----+



-------------------------------------------
Batch: 1
-------------------------------------------
+---+-----------------+-----+------+-----+-------+---+----+
|cid|currentInstructor|major|number|title|courses| id|name|
+---+-----------------+-----+------+-----+-------+---+----+
+---+-----------------+-----+------+-----+-------+---+----+



-------------------------------------------
Batch: 2
-------------------------------------------
+---+-----------------+-----+------+--------------------+-------+---+-----------------+
|cid|currentInstructor|major|number|               title|courses| id|             name|
+---+-----------------+-----+------+--------------------+-------+---+-----------------+
|  3|                0| MIST|  5730|Advanced Data Man...| [0, 3]|  0|      Hani Safadi|
|  2|                1| MIST|  4630|Network-Based App...|    [2]|  1|     Craig Piercy|
|  1|                3| MIST|  4610|Data Managment an...| [0, 1]|  3|Nikhil Srinivasan|
|  0|                4| MIST|  4600|Computer Programm...|    [0]|  4|      Karen Aguar|
+---+-----------------+-----+------+--------------------+-------+---+-----------------+

Stream-stream join complete


# Shutdown Spark when done

In [13]:
spark.stop()